# 01 — Feature Extraction
**Sign2Chat — UAE Sign Language Recognition**

Extracts MediaPipe Holistic landmarks from every video and saves them as `.npy` files,
then runs data augmentation on the training split.

### Key config
- `MAX_FRAMES = 180` @ 30fps = **6 seconds** — covers signs up to 4 seconds with buffer
- Start with **55 words** to validate the pipeline, scale to 100 for the viva
- Change `SUBSET_SIZE = 55` → `100` when ready

### Output per video — shape `(180, 165)`
| Source | Nodes | Features |
|---|---|---|
| Left hand | 21 × (x,y,z) | 63 |
| Right hand | 21 × (x,y,z) | 63 |
| Pose (9 key joints) | 9 × (x,y,z) | 27 |
| Face (4 reference points) | 4 × (x,y,z) | 12 |
| **Total** | **55 nodes** | **165** |

### Files produced
| File | Description |
|---|---|
| `processed/<folder>/<video>.npy` | Shape `(180, 165)` float32 |
| `processed/augmented/...` | Augmented copies (train only) |
| `landmarks_index.csv` | Index of all original .npy files |
| `landmarks_index_augmented.csv` | Index including augmented samples |

In [1]:
!pip install "mediapipe>=0.10.0" opencv-python-headless pandas numpy tqdm scikit-learn

Defaulting to user installation because normal site-packages is not writeable


## Step 1 — Imports & Config

In [2]:
import os, json
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import HolisticLandmarker, HolisticLandmarkerOptions
from mediapipe.tasks.python.vision.core.vision_task_running_mode import VisionTaskRunningMode
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── PATHS ─────────────────────────────────────────────────────────────────────
VIDEOS_ROOT    = '../UAE-dataset/UAE100'                  # root folder — subfolders are labels
CLASS_MAP_PATH = '../UAE-dataset/class_map.json'  # label_id → clean_name mapping
OUTPUT_DIR     = '../UAE-dataset/processed'
INDEX_CSV      = '../UAE-dataset/landmarks_index.csv'
MP_MODEL_PATH  = '../models/holistic_landmarker.task'

# ── Sequence config ────────────────────────────────────────────────────────────
MAX_FRAMES  = 180    # 6 seconds @ 30fps — covers signs up to 4s with buffer
FEATURE_DIM = 165    # 55 nodes × 3 coords (x, y, z)

# ── Subset size ────────────────────────────────────────────────────────────────
# 55  → pipeline validation (use now)
# 100 → full dataset for viva (change when ready)
SUBSET_SIZE = 34

# ── Train / Val / Test split ratios (applied per class) ───────────────────────
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
RANDOM_SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'augmented'), exist_ok=True)
os.makedirs('../models', exist_ok=True)

print(f'✅ Config ready')
print(f'   MAX_FRAMES  = {MAX_FRAMES}  ({MAX_FRAMES/30:.1f} seconds @ 30fps)')
print(f'   FEATURE_DIM = {FEATURE_DIM}')
print(f'   SUBSET_SIZE = {SUBSET_SIZE}  words')
print(f'   VIDEOS_ROOT = {VIDEOS_ROOT}')
print(f'   mediapipe   = {mp.__version__}')

I0000 00:00:1780830649.840333   16737 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780830649.881928   16737 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780830651.908735   16737 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Config ready
   MAX_FRAMES  = 180  (6.0 seconds @ 30fps)
   FEATURE_DIM = 165
   SUBSET_SIZE = 34  words
   VIDEOS_ROOT = ../UAE-dataset/UAE100
   mediapipe   = 0.10.35


## Step 2 — Load Class Map & Discover Subfolders

In [3]:
# ── Load class map (label_id → clean_name) ───────────────────────────────────
with open(CLASS_MAP_PATH) as f:
    class_map = json.load(f)
# class_map["34"] = {label_id, word, clean_name, folder}
# folder value matches the subfolder name: "34_Cat"

# Build lookup: folder_name → class_map entry
folder_to_class = {v['folder']: v for v in class_map.values()}

print(f'✅ Class map loaded: {len(class_map)} total classes')
print(f'   Using {SUBSET_SIZE} for this run')

# ── Discover subfolders under VIDEOS_ROOT ─────────────────────────────────────
if not os.path.exists(VIDEOS_ROOT):
    print(f'\n⚠️  VIDEOS_ROOT not found: {VIDEOS_ROOT}')
    print(f'   Create it and place your word subfolders inside.')
    raise FileNotFoundError(VIDEOS_ROOT)

subfolders = sorted([
    d for d in os.listdir(VIDEOS_ROOT)
    if os.path.isdir(os.path.join(VIDEOS_ROOT, d))
])

print(f'\n✅ Found {len(subfolders)} subfolders under {VIDEOS_ROOT}/')
print(f'   First 5: {subfolders[:5]}')

# ── Match subfolders to class map ─────────────────────────────────────────────
matched, unmatched = [], []
for folder in subfolders:
    if folder in folder_to_class:
        matched.append(folder)
    else:
        unmatched.append(folder)

if unmatched:
    print(f'\n⚠️  {len(unmatched)} subfolders not found in class_map:')
    for f in unmatched[:5]:
        print(f'   {f}')

print(f'\n✅ Matched {len(matched)} subfolders to class map')

# ── Select SUBSET_SIZE classes ─────────────────────────────────────────────────
# Sort matched folders by label_id for reproducibility
matched_sorted = sorted(matched, key=lambda f: folder_to_class[f]['label_id'])
selected_folders = matched_sorted[:SUBSET_SIZE]

print(f'\n=== Using {len(selected_folders)} classes ===')
for folder in selected_folders[:10]:
    info = folder_to_class[folder]
    print(f'  label_id={info["label_id"]:4d}  {folder}')
if len(selected_folders) > 10:
    print(f'  ... and {len(selected_folders) - 10} more')

✅ Class map loaded: 100 total classes
   Using 34 for this run

✅ Found 34 subfolders under ../UAE-dataset/UAE100/
   First 5: ['104_Slow', '107_Fatigue', '109_Hungry', '112_Beautiful', '113_Hot']

✅ Matched 34 subfolders to class map

=== Using 34 classes ===
  label_id=  34  34_Cat
  label_id=  38  38_Elephant
  label_id=  52  52_Fish
  label_id=  56  56_Camel
  label_id= 104  104_Slow
  label_id= 107  107_Fatigue
  label_id= 109  109_Hungry
  label_id= 112  112_Beautiful
  label_id= 113  113_Hot
  label_id= 115  115_Sad
  ... and 24 more


## Step 3 — Discover All Videos

In [4]:
# ── Collect all .mp4 files from selected subfolders ──────────────────────────
# Video filenames are ignored — the subfolder name is the label.

video_records = []

for folder in selected_folders:
    info       = folder_to_class[folder]
    folder_path = os.path.join(VIDEOS_ROOT, folder)

    mp4_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith('.mp4')
    ])

    if len(mp4_files) == 0:
        print(f'⚠️  No videos in {folder}')
        continue

    for fname in mp4_files:
        video_records.append({
            'video_id':   f"{folder}__{fname.replace('.mp4','')}",  # unique id
            'label_id':   info['label_id'],
            'clean_name': info['clean_name'],
            'folder':     folder,
            'video_path': os.path.join(folder_path, fname),
            'filename':   fname,
        })

df_all = pd.DataFrame(video_records)

print(f'✅ Found {len(df_all)} videos across {df_all["label_id"].nunique()} classes')
print(f'\n=== Videos per class ===')
per_class = df_all.groupby('clean_name').size().sort_values()
print(f'  Min : {per_class.min()} ({per_class.idxmin()})')
print(f'  Max : {per_class.max()} ({per_class.idxmax()})')
print(f'  Avg : {per_class.mean():.1f}')
print(f'\nClasses with fewest videos:')
print(per_class.head(5).to_string())

✅ Found 1239 videos across 34 classes

=== Videos per class ===
  Min : 31 (White)
  Max : 44 (Scared)
  Avg : 36.4

Classes with fewest videos:
clean_name
White      31
Necktie    33
Red        34
Walk       34
Write      34


## Step 4 — Train / Val / Test Split

Split is done **per class** (stratified) so every class is represented in every split.
With ~50 videos per class and 70/15/15 split:
- Train : ~35 videos/class  (+augmentation → ~385/class)
- Val   :  ~7 videos/class
- Test  :  ~7 videos/class

In [5]:
split_records = []

for label_id, group in df_all.groupby('label_id'):
    group = group.copy().reset_index(drop=True)
    n = len(group)

    if n < 3:
        # Too few — all go to train
        group['split'] = 'train'
        split_records.append(group)
        print(f'⚠️  Only {n} videos for label_id={label_id} — all assigned to train')
        continue

    try:
        train_idx, temp_idx = train_test_split(
            group.index.tolist(),
            test_size    = VAL_RATIO + TEST_RATIO,
            random_state = RANDOM_SEED
        )
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size    = TEST_RATIO / (VAL_RATIO + TEST_RATIO),
            random_state = RANDOM_SEED
        )
    except ValueError:
        train_idx = group.index.tolist()
        val_idx, test_idx = [], []

    split_map = (
        {i: 'train' for i in train_idx} |
        {i: 'val'   for i in val_idx}   |
        {i: 'test'  for i in test_idx}
    )
    group['split'] = group.index.map(split_map).fillna('train')
    split_records.append(group)

df = pd.concat(split_records, ignore_index=True)

print('=== Split distribution ===')
print(df['split'].value_counts().to_string())
print(f'\nPer-class split (first 5 classes):')
print(df.groupby(['clean_name','split']).size().unstack(fill_value=0).head(5).to_string())

=== Split distribution ===
split
train    850
test     203
val      186

Per-class split (first 5 classes):
split       test  train  val
clean_name                  
Angry          6     25    5
Beautiful      6     27    6
Black          6     25    6
Blue           6     25    5
Camel          6     26    6


## Step 5 — Landmark Extraction Functions

In [6]:
POSE_INDICES = [0, 11, 12, 13, 14, 15, 16, 23, 24]  # 9 upper-body joints
FACE_INDICES = [152, 10, 234, 454]                   # 4 face reference points

def lm_to_arr(lm_list, indices=None, n=21):
    """MediaPipe landmark list → (N, 3) float32, normalized to [-1, 1].
    Returns zeros if not detected.
    """
    if not lm_list:
        k = len(indices) if indices is not None else n
        return np.zeros((k, 3), dtype=np.float32)
    arr = np.array([[l.x, l.y, l.z] for l in lm_list], dtype=np.float32)
    if indices is not None:
        arr = arr[indices]
    arr[:, :2] = 2.0 * (arr[:, :2] - 0.5)   # [0,1] → [-1,1]
    return arr


def extract_frame_features(result):
    """One HolisticLandmarkerResult → (165,) float32 normalized feature vector.

    Layout: [lh(63) | rh(63) | pose(27) | face(12)]
    All coordinates normalized relative to shoulder midpoint.
    """
    lh   = lm_to_arr(result.left_hand_landmarks,  n=21)
    rh   = lm_to_arr(result.right_hand_landmarks, n=21)
    pose = lm_to_arr(result.pose_landmarks,  POSE_INDICES, 33)
    face = lm_to_arr(result.face_landmarks,  FACE_INDICES, 468)

    # Shoulder midpoint: pose[1]=L_shoulder, pose[2]=R_shoulder
    origin = (pose[1] + pose[2]) / 2.0 if result.pose_landmarks              else np.zeros(3, dtype=np.float32)

    return np.concatenate([
        (lh   - origin).flatten(),
        (rh   - origin).flatten(),
        (pose - origin).flatten(),
        (face - origin).flatten(),
    ]).astype(np.float32)   # (165,)


def pad_or_truncate(seq, max_frames=MAX_FRAMES):
    """(T, 165) → (MAX_FRAMES, 165).

    T < MAX_FRAMES : zero-pad at the FRONT
                     (Masking layer ignores leading zeros — sign at the end)
    T > MAX_FRAMES : keep the LAST max_frames frames
                     (sign is usually in the middle/end of the clip)
    """
    T, D = seq.shape
    if T >= max_frames:
        return seq[-max_frames:]
    return np.vstack([np.zeros((max_frames - T, D), dtype=np.float32), seq])


def extract_video(video_path, landmarker, ts_counter):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        # Jump counter forward even for empty videos
        ts_counter[0] += 10000
        return None

    vecs = []
    success = True

    for _ in range(total):
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (640, 480))
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mpi   = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts    = ts_counter[0]
        ts_counter[0] += 1

        try:
            res = landmarker.detect_for_video(mpi, ts)
            vecs.append(extract_frame_features(res))
        except (RuntimeError, ValueError):
            success = False
            break

    cap.release()

    # ── Always jump forward by a large gap after each video ──────────────────
    # This guarantees the next video's first timestamp is always
    # ahead of MediaPipe's internal state, regardless of what happened.
    ts_counter[0] += 10000

    if not success or not vecs:
        return None

    return pad_or_truncate(np.array(vecs, dtype=np.float32))


print(f'✅ Functions defined')
print(f'   MAX_FRAMES  = {MAX_FRAMES}  ({MAX_FRAMES/30:.1f}s @ 30fps)')
print(f'   FEATURE_DIM = {FEATURE_DIM}')

✅ Functions defined
   MAX_FRAMES  = 180  (6.0s @ 30fps)
   FEATURE_DIM = 165


## Step 6 — Extract All Videos

In [8]:
import builtins
if not hasattr(builtins, '_ts_counter'):
    builtins._ts_counter = [0]
ts_counter = builtins._ts_counter

options = HolisticLandmarkerOptions(
    base_options = mp_python.BaseOptions(model_asset_path=MP_MODEL_PATH),
    running_mode = VisionTaskRunningMode.VIDEO,
    min_face_detection_confidence  = 0.5,
    min_face_landmarks_confidence  = 0.5,
    min_pose_detection_confidence  = 0.5,
    min_pose_landmarks_confidence  = 0.5,
    min_hand_landmarks_confidence  = 0.5,
    output_face_blendshapes        = False,
    output_segmentation_mask       = False,
)

for folder in selected_folders:
    os.makedirs(os.path.join(OUTPUT_DIR, folder), exist_ok=True)

index_records = []
skipped       = []
landmarker    = HolisticLandmarker.create_from_options(options)

for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting'):
    out_path = os.path.join(OUTPUT_DIR, row['folder'],
                            f"{row['video_id']}.npy")

    if os.path.exists(out_path):
        index_records.append({**row.to_dict(), 'npy_path': out_path})
        continue

    seq = extract_video(row['video_path'], landmarker, ts_counter)

    if seq is None:
        # Recreate landmarker with fresh state after any failure
        try:
            landmarker.close()
        except Exception:
            pass
        landmarker = HolisticLandmarker.create_from_options(options)
        ts_counter[0] += 10000   # extra jump after recreating
        skipped.append(row['video_id'])
        print(f'\n⚠️  Skipped {row["video_id"]}')
        continue

    np.save(out_path, seq)
    index_records.append({**row.to_dict(), 'npy_path': out_path})

# Clean up
try:
    landmarker.close()
except Exception:
    pass

df_index = pd.DataFrame(index_records)
df_index.to_csv(INDEX_CSV, index=False)

print(f'\n✅ Extraction done')
print(f'   Processed : {len(index_records)}')
print(f'   Skipped   : {len(skipped)}')
print(f'   ts_counter: {ts_counter[0]}')
if skipped:
    print(f'   Skipped files: {skipped}')
print(f'\n=== Split distribution ===')
print(df_index['split'].value_counts().to_string())

I0000 00:00:1780830761.675077   17191 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1780830761.698118   17214 gl_context.cc:385] GL version: 3.1 (OpenGL ES 3.1 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: D3D12 (NVIDIA GeForce RTX 3080)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780830761.753945   17196 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780830761.777984   17197 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780830761.781696   17203 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780830761.782204   17195 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signa


✅ Extraction done
   Processed : 1239
   Skipped   : 0
   ts_counter: 0

=== Split distribution ===
split
train    850
test     203
val      186


## Step 7 — Augmentation (Training Split Only)

7 geometry-only transforms. Applied to training videos only.
Val and test are never augmented — they must reflect real-world conditions.

| # | Transform | Parameters | Purpose |
|---|---|---|---|
| 1 | Rotation | ±20° | Camera angle variation |
| 2 | Scale | ±20% | Hand size variation |
| 3 | Noise | σ = 0.02 | Natural hand tremor |
| 4 | Time warp | Drop 20% frames, prob=0.7 | Signing speed variation |
| 5 | Temporal shift | ±5 frames | Timing offset variation |
| 6 | Hand occlusion | 1–4 frames, prob=0.5 | Partial visibility |

In [9]:
def augment_sequence(seq):
    """Random geometry-only augmentation on (MAX_FRAMES, 165) sequence."""
    aug = seq.copy()

    # 2. Rotation ±20°
    a    = np.random.uniform(-20, 20) * np.pi / 180
    c, s = np.cos(a), np.sin(a)
    xs   = aug[:, 0::3].copy()
    ys   = aug[:, 1::3].copy()
    aug[:, 0::3] = c * xs - s * ys
    aug[:, 1::3] = s * xs + c * ys

    # 3. Scale ±20%
    aug *= np.random.uniform(0.8, 1.2)

    # 4. Noise σ=0.02
    aug += np.random.normal(0, 0.02, aug.shape).astype(np.float32)

    # 5. Time warp — drop up to 20% of active frames, then re-pad
    if np.random.rand() < 0.7:
        active = np.where(np.any(aug != 0, axis=1))[0]
        if len(active) > 10:
            n_drop = max(1, len(active) // 5)
            drop   = np.random.choice(active, n_drop, replace=False)
            aug    = np.delete(aug, drop, axis=0)
            aug    = pad_or_truncate(aug)

    # 6. Temporal shift ±5 frames
    shift = np.random.randint(-5, 6)
    if shift > 0:
        aug = np.vstack([np.zeros((shift, aug.shape[1]),  dtype=np.float32), aug[:-shift]])
    elif shift < 0:
        aug = np.vstack([aug[-shift:], np.zeros((-shift, aug.shape[1]), dtype=np.float32)])

    # 7. Hand occlusion
    if np.random.rand() < 0.5:
        n_frames = np.random.randint(1, 5)
        zf       = np.random.choice(MAX_FRAMES, n_frames, replace=False)
        if np.random.rand() < 0.5:
            aug[zf,  0:63]  = 0   # left hand
        else:
            aug[zf, 63:126] = 0   # right hand

    return aug.astype(np.float32)


# Sanity check
_test = np.load(df_index.iloc[0]['npy_path'])
_aug  = augment_sequence(_test)
assert _aug.shape == (MAX_FRAMES, FEATURE_DIM)
assert not np.array_equal(_test, _aug)
print('✅ augment_sequence verified')
print(f'   Input  shape: {_test.shape}  range [{_test.min():.3f}, {_test.max():.3f}]')
print(f'   Output shape: {_aug.shape}  range [{_aug.min():.3f}, {_aug.max():.3f}]')

✅ augment_sequence verified
   Input  shape: (180, 165)  range [-1.117, 1.487]
   Output shape: (180, 165)  range [-1.331, 1.734]


In [10]:
AUG_COPIES     = 10
AUG_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'augmented')

# Mirror augmented folder structure
for folder in selected_folders:
    os.makedirs(os.path.join(AUG_OUTPUT_DIR, folder), exist_ok=True)

train_df    = df_index[df_index['split'] == 'train'].reset_index(drop=True)
aug_records = []

print(f'Augmenting {len(train_df)} training videos × {AUG_COPIES} copies ...')

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Augmenting'):
    seq = np.load(row['npy_path'])
    for i in range(AUG_COPIES):
        aug_seq  = augment_sequence(seq)
        aug_path = os.path.join(AUG_OUTPUT_DIR, row['folder'],
                                f"{row['video_id']}_aug{i:02d}.npy")
        np.save(aug_path, aug_seq)
        aug_records.append({
            'video_id':   f"{row['video_id']}_aug{i:02d}",
            'label_id':   row['label_id'],
            'clean_name': row['clean_name'],
            'folder':     row['folder'],
            'split':      'train',
            'npy_path':   aug_path,
        })

FULL_INDEX_CSV = '../UAE-dataset/landmarks_index_augmented.csv'
df_aug  = pd.DataFrame(aug_records)
df_full = pd.concat([df_index, df_aug], ignore_index=True)
df_full.to_csv(FULL_INDEX_CSV, index=False)

print(f'\n✅ Augmentation done')
print(f'   Original train  : {len(train_df)}')
print(f'   Augmented train : {len(aug_records)}  ({AUG_COPIES}× each)')
print(f'   Total dataset   : {len(df_full)}')
print(f'\n=== Final split counts ===')
print(df_full['split'].value_counts().to_string())
print(f'\nTraining samples per class (avg): {len(df_full[df_full["split"]=="train"]) / SUBSET_SIZE:.0f}')

Augmenting 850 training videos × 10 copies ...


Augmenting: 100%|██████████| 850/850 [00:11<00:00, 76.45it/s]


✅ Augmentation done
   Original train  : 850
   Augmented train : 8500  (10× each)
   Total dataset   : 9739

=== Final split counts ===
split
train    9350
test      203
val       186

Training samples per class (avg): 275


## Step 9 — Final Summary

In [11]:
print('=' * 55)
print('FEATURE EXTRACTION COMPLETE')
print('=' * 55)

tr  = df_full[df_full['split'] == 'train']
val = df_full[df_full['split'] == 'val']
tst = df_full[df_full['split'] == 'test']

print(f'\n  Words        : {SUBSET_SIZE}  (set SUBSET_SIZE=100 for full run)')
print(f'  MAX_FRAMES   : {MAX_FRAMES}  ({MAX_FRAMES/30:.1f}s @ 30fps)')
print(f'  FEATURE_DIM  : {FEATURE_DIM}  (55 nodes × 3 coords)')
print(f'\n  Split         Samples   Per class')
print(f'  ──────────── ─────────  ─────────')
print(f'  train        {len(tr):6}   {len(tr)/SUBSET_SIZE:.0f}  (incl. {AUG_COPIES}× augmentation)')
print(f'  val          {len(val):6}   {len(val)/SUBSET_SIZE:.1f}')
print(f'  test         {len(tst):6}   {len(tst)/SUBSET_SIZE:.1f}')
print(f'  ──────────── ─────────')
print(f'  TOTAL        {len(df_full):6}')

print(f'\n  Normalization : shoulder-midpoint, coordinates in [-1, 1]')
print(f'  Padding       : zeros at front, sign at end of sequence')
print(f'  Augmentation  : train only — 7 geometry-only transforms')

print(f'\n  Output files:')
print(f'    {INDEX_CSV}')
print(f'    {FULL_INDEX_CSV}')
print(f'    {OUTPUT_DIR}/<folder>/*.npy')
print(f'    {AUG_OUTPUT_DIR}/<folder>/*.npy')

# Spot check one sample per split
print(f'\n=== Spot check ===')
for split in ['train', 'val', 'test']:
    row = df_full[df_full['split'] == split].iloc[0]
    seq = np.load(row['npy_path'])
    nz  = np.any(seq != 0, axis=1).sum()
    print(f'  {split:<6} {row["clean_name"]:<25} '
          f'shape={seq.shape}  active={nz}/{MAX_FRAMES}  '
          f'range=[{seq.min():.3f},{seq.max():.3f}]')

print(f'\n✅ Ready for notebook 02 — model training')
print(f'   To scale to 100 words: set SUBSET_SIZE = 100 in Step 1 and re-run')

FEATURE EXTRACTION COMPLETE

  Words        : 34  (set SUBSET_SIZE=100 for full run)
  MAX_FRAMES   : 180  (6.0s @ 30fps)
  FEATURE_DIM  : 165  (55 nodes × 3 coords)

  Split         Samples   Per class
  ──────────── ─────────  ─────────
  train          9350   275  (incl. 10× augmentation)
  val             186   5.5
  test            203   6.0
  ──────────── ─────────
  TOTAL          9739

  Normalization : shoulder-midpoint, coordinates in [-1, 1]
  Padding       : zeros at front, sign at end of sequence
  Augmentation  : train only — 7 geometry-only transforms

  Output files:
    ../UAE-dataset/landmarks_index.csv
    ../UAE-dataset/landmarks_index_augmented.csv
    ../UAE-dataset/processed/<folder>/*.npy
    ../UAE-dataset/processed/augmented/<folder>/*.npy

=== Spot check ===
  train  Cat                       shape=(180, 165)  active=122/180  range=[-1.117,1.487]
  val    Cat                       shape=(180, 165)  active=122/180  range=[-1.154,1.477]
  test   Cat            